In [64]:
#import

from langgraph.graph import StateGraph, START, END
from typing import TypedDict 

from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

In [65]:
load_dotenv 

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [66]:

model = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0
)

In [78]:
# define a state
class BlogState(TypedDict):
    content: str
    topic: str
    outline: str
    rating: int


In [79]:
def create_outline(state: BlogState) -> BlogState:
    # Fetch Topic
    topic = state['topic']

    #call llm gen outline
    prompt = f'generate a detailed outline for a blog on the topic- {topic}'
    outline = model.invoke(prompt).content

    #update state
    state['outline'] = outline

    return state

In [80]:
def create_blog(state: BlogState) -> BlogState:
    topic = state['topic']
    outline = state['outline']

    prompt = f'write a detailed on the topic - {topic} using the following outline \n {outline}'

    content = model.invoke(prompt).content
    state['content']= content

    return state

In [ ]:
def evaluate(state: BlogState) -> BlogState:
    #based on the outline rate the blog

    #fetch
    outline = state['outline']
    content = state['content']

    prompt = f'based on the outline- {outline} rate the following blog on the scale of 1 to 10  \n {content}'

    # call llm
    rating = model.invoke(prompt)

    state['rating'] = int(rating.content.strip())


    return state


    

In [82]:
# define a graph
graph = StateGraph(BlogState)

# add nodes to the graph
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('evaluate', evaluate)

# add edges to the graph
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate')
graph.add_edge('evaluate', END)

# compile the graph
workflow = graph.compile()



In [83]:
#execute the graph
initial_state = {'topic': 'the importance of the water in human body'}
final_state = workflow.invoke(initial_state)
print(final_state)

ChatGoogleGenerativeAIError: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 46.126623147s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-Fr

In [ ]:
print(final_state['outline'])

[{'type': 'text', 'text': 'Here is a comprehensive, SEO-optimized, and detailed outline for a blog post on **"The Importance of Water in the Human Body."**\n\n---\n\n# Blog Title Ideas:\n* **Option 1:** The Elixir of Life: Why Water is the Ultimate Fuel for the Human Body\n* **Option 2:** H2O & You: The Science-Backed Importance of Staying Hydrated\n* **Option 3:** More Than Just Thirst: 7 Vital Functions of Water in Your Body\n* **Option 4:** The Ultimate Guide to Hydration: How Water Controls Your Health, Mind, and Energy\n\n---\n\n## I. Introduction\n* **Catchy Hook:** Start with a mind-blowing statistic (e.g., *"Up to 60% of the adult human body is water—meaning you are literally a walking body of water."*)\n* **The Problem/Reality:** Many people treat water as an afterthought, relying instead on coffee, soda, or energy drinks, leading to chronic mild dehydration.\n* **Thesis Statement:** Water is not just a drink; it is a vital nutrient necessary for cellular function, energy prod

In [ ]:
print(final_state['content'])

[{'type': 'text', 'text': '# The Elixir of Life: Why Water is the Ultimate Fuel for the Human Body\n\n## I. Introduction\n\nUp to **60% of the adult human body is made of water**—meaning you are, quite literally, a walking body of water. Yet despite its fundamental role in our survival, many people treat water as an afterthought. Busy work schedules, endless distractions, and the widespread availability of coffee, sodas, and energy drinks often lead to chronic mild dehydration without us even realizing it. \n\nWater is far more than a simple beverage to quench a dry throat; it is an essential nutrient required by every single cell, tissue, and biological process in your body. From driving energy production and mental clarity to enabling seamless digestion and joint health, proper hydration is the bedrock of optimal physical and cognitive function. \n\nIn this ultimate guide to hydration, we will explore the science-backed importance of water, break down how it functions across your maj

In [ ]:
print(final_state['rating'])

content=[{'type': 'text', 'text': 'I would rate this blog post a **9.5 out of 10**. \n\nHere is a breakdown of the rating based on structure, adherence to the outline, readability, and content quality:\n\n---\n\n### **Strengths (Why it scores so high):**\n\n1. **Flawless Outline Adherence (10/10):**\n   * The post follows every single section, subhead, statistic, and key point specified in the outline without skipping any required details.\n   * It seamlessly integrates all required sections: biology, top 6 functions, physical/aesthetic benefits, dehydration signs, recommendations/myth-busting, and actionable tips.\n\n2. **Formatting & Visual Engagement (9.5/10):**\n   * **ASCII Diagrams/Tables:** The addition of structured ASCII boxes (e.g., for the top functions, dehydration spectrum, and practical daily strategy) elevates the visual presentation, making complex information scannable and easy to digest.\n   * **Clear Hierarchy:** The use of Markdown headers (`#`, `##`, `###`), bold t